In [1]:
import time
import logging
import pandas as pd
from datetime import datetime
from google.cloud import bigquery, storage


# récupère moi le csv meteo_paris.csv dans le dossier meteo et le charge dans un dataframe pandas
def load_meteo_data(city_name: str) -> pd.DataFrame:
    """
    Charge les données météorologiques à partir d'un fichier CSV.

    Args:
        city_name: Nom de la ville pour laquelle les données sont chargées

    Returns:
        DataFrame contenant les données météorologiques
    """
    file_path = f"/home/merville.thibaul/code/ThibaultMer/mix-energy/data/meteo/meteo_{city_name}.csv"
    df = pd.read_csv(file_path)
    return df


df = load_meteo_data("paris")

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 264 entries, 0 to 263
Data columns (total 28 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   time                       264 non-null    str    
 1   temperature_2m             264 non-null    float64
 2   relative_humidity_2m       264 non-null    int64  
 3   dew_point_2m               264 non-null    float64
 4   precipitation_probability  264 non-null    int64  
 5   precipitation              264 non-null    float64
 6   rain                       264 non-null    float64
 7   showers                    264 non-null    float64
 8   snowfall                   264 non-null    float64
 9   snow_depth                 264 non-null    float64
 10  weather_code               264 non-null    int64  
 11  pressure_msl               264 non-null    float64
 12  surface_pressure           264 non-null    float64
 13  cloud_cover                264 non-null    int64  
 14  evapo

In [3]:
def create_bigquery_schema(df: pd.DataFrame) -> list[bigquery.SchemaField]:
    """
    Crée un schéma compatible BigQuery à partir d'un DataFrame pandas.

    Args:
        df: DataFrame pour lequel le schéma doit être créé

    Returns:
        Liste de champs BigQuery (SchemaField)
    """
    schema = []
    for column_name, dtype in df.dtypes.items():
        if column_name == "time":
            bigquery_type = "TIMESTAMP"
        elif column_name == "date":
            bigquery_type = "DATE"
        elif column_name == "heure":
            bigquery_type = "DATETIME"
        elif column_name == "date_heure":
            bigquery_type = "TIMESTAMP"
        elif pd.api.types.is_bool_dtype(dtype):
            bigquery_type = "BOOLEAN"
        elif pd.api.types.is_integer_dtype(dtype):
            bigquery_type = "INTEGER"
        elif pd.api.types.is_float_dtype(dtype):
            bigquery_type = "FLOAT"
        elif pd.api.types.is_datetime64_any_dtype(dtype):
            bigquery_type = "TIMESTAMP"
        elif pd.api.types.is_string_dtype(dtype):
            bigquery_type = "STRING"
        else:
            bigquery_type = "STRING"

        schema.append(bigquery.SchemaField(name=str(column_name), field_type=bigquery_type, mode="NULLABLE"))
    return schema

In [4]:
schema = create_bigquery_schema(df)
schema

[SchemaField('time', 'TIMESTAMP', 'NULLABLE', None, None, (), None, None),
 SchemaField('temperature_2m', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('relative_humidity_2m', 'INTEGER', 'NULLABLE', None, None, (), None, None),
 SchemaField('dew_point_2m', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('precipitation_probability', 'INTEGER', 'NULLABLE', None, None, (), None, None),
 SchemaField('precipitation', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('rain', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('showers', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('snowfall', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('snow_depth', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('weather_code', 'INTEGER', 'NULLABLE', None, None, (), None, None),
 SchemaField('pressure_msl', 'FLOAT', 'NULLABLE', None, None, (), None, None),
 SchemaField('surface_pressure', 'FLOAT', 'NU